In [14]:
import pandas as pd
import numpy as np
from contourpy.util.data import simple
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.tree import DecisionTreeClassifier

In [2]:
df = pd.read_csv("train.csv")

In [3]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"], inplace=True)

In [6]:
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=["Survived"]), df["Survived"], test_size=0.2, random_state=42)

In [7]:
X_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
331,1,male,45.5,0,0,28.5000,S
733,2,male,23.0,0,0,13.0000,S
382,3,male,32.0,0,0,7.9250,S
704,3,male,26.0,1,0,7.8542,S
813,3,female,6.0,4,2,31.2750,S


In [8]:
y_train.sample(5)

519    0
156    1
823    1
651    1
553    1
Name: Survived, dtype: int64

In [11]:
X_train.isnull().sum()

Pclass        0
Sex           0
Age         140
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

In [15]:
#imputation transformer
trf1 = ColumnTransformer(
    [
        ("impute_age", SimpleImputer(),[2]), # where [2] is col index
        ("impute_embarked", SimpleImputer(strategy= 'most_frequent'),[6]),
    ], remainder="passthrough"
)

In [16]:
#OneHotEncoding

trf2 = ColumnTransformer(
    [
        ('oho_sex_embarked', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), [1,6]),
    ], remainder="passthrough"
)

In [17]:
trf3 = ColumnTransformer(
    [
        ('scale', MinMaxScaler(), slice(0,10))
    ]
)

In [18]:
trf4 = DecisionTreeClassifier()

In [19]:
pipe = Pipeline(
    [
        ("trf1", trf1),
        ("trf2", trf2),
        ("trf3", trf3),
        ("trf4", trf4),
    ]
)

# My personal Change for better prediction

In [25]:
preprocessor = ColumnTransformer(
    [
        ('impute_age', SimpleImputer(strategy='mean'), [2]),
        ('embarked', Pipeline([
            ("impute", SimpleImputer(strategy='most_frequent')),
            ("ohe", OneHotEncoder(sparse_output=False, handle_unknown='ignore')),
        ]), ['Embarked']),
        ('sex', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), ['Sex']),
        ('scale', MinMaxScaler(), ['Pclass', 'SibSp', "Parch", "Fare"]),
    ], remainder="drop"
)

In [26]:
pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("clf", DecisionTreeClassifier(random_state=42)),
])

In [27]:
pipe.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('impute_age', ...), ('embarked', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [28]:
pipe.named_steps

{'preprocessor': ColumnTransformer(transformers=[('impute_age', SimpleImputer(), [2]),
                                 ('embarked',
                                  Pipeline(steps=[('impute',
                                                   SimpleImputer(strategy='most_frequent')),
                                                  ('ohe',
                                                   OneHotEncoder(handle_unknown='ignore',
                                                                 sparse_output=False))]),
                                  ['Embarked']),
                                 ('sex',
                                  OneHotEncoder(handle_unknown='ignore',
                                                sparse_output=False),
                                  ['Sex']),
                                 ('scale', MinMaxScaler(),
                                  ['Pclass', 'SibSp', 'Parch', 'Fare'])]),
 'clf': DecisionTreeClassifier(random_state=42)}

In [29]:
y_pred = pipe.predict(X_test)

In [30]:
y_pred

array([0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0,
       0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1,
       0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1,
       0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0,
       1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0,
       0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0,
       0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0,
       0, 1, 1])

In [31]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test, y_pred)

0.7877094972067039

In [32]:
import pickle
pickle.dump(pipe, open('pipe.pkl', 'wb'))